# Vector Spaces

## What's covered

- What a vector space is — the rules vectors must obey
- **Span** — the set of all linear combinations of a group of vectors
- **Linear independence** vs **linear dependence** — when do vectors carry redundant information
- **Basis** — the smallest set that spans the space
- **Dimension** — the size of any basis, an intrinsic property of the space
- **Subspaces** — vector spaces sitting inside larger vector spaces
- Where this appears in ML — feature redundancy, latent dimensions, PCA's directions as a basis


## What is a vector space?

In the previous notebook we used two operations on vectors: **addition** and **scalar multiplication**. A **vector space** is just a set of objects (vectors) that is *closed* under these two operations — meaning that if you take any two vectors in the set and add them, you stay in the set; and if you scale any vector in the set by a number, you also stay in the set.

The canonical vector space is `R^n` — all real-valued lists of length `n`. `R^2` is the plane, `R^3` is 3D space, `R^784` is the space MNIST images live in.

This sounds abstract, but the payoff is concrete: once we know we are inside a vector space, we get to ask the three questions this notebook is about:

1. *What can I reach by combining these vectors?* — the **span**
2. *Are any of these vectors redundant?* — **linear independence**
3. *What is the minimum number of vectors I need to describe everything?* — **basis** and **dimension**

These three questions are the entire vocabulary of feature engineering, embedding design, and dimensionality reduction in ML.


## Span — what you can reach

The **span** of a set of vectors is the set of *all linear combinations* of those vectors.

If you have a single vector **v**, its span is the **line** through the origin in the direction of **v**:

$$
\text{span}\{\mathbf{v}\} = \{ c \mathbf{v} \;|\; c \in \mathbb{R} \}
$$

If you have two vectors **u** and **v** that point in different directions, their span is the entire **plane** through the origin containing both:

$$
\text{span}\{\mathbf{u}, \mathbf{v}\} = \{ a \mathbf{u} + b \mathbf{v} \;|\; a, b \in \mathbb{R} \}
$$

But if **u** and **v** point in the *same* direction (one is a scalar multiple of the other), their span is still just a line. **Adding a redundant vector doesn't enlarge the span.**

In R³, three vectors usually span all of R³ (the whole 3D space). But if all three happen to lie in the same plane, their span is just that plane.

This is the central question of span: *given these vectors, what is the geometric shape — line, plane, full space — that I can reach with them?*


In [ ]:
import numpy as np

# Standard basis vectors of R^2
e1 = np.array([1, 0])
e2 = np.array([0, 1])

# A few linear combinations — any point in R^2 is reachable
print("3*e1 + 2*e2 =", 3*e1 + 2*e2)   # (3, 2)
print("-1*e1 + 5*e2 =", -1*e1 + 5*e2) # (-1, 5)

# A redundant pair — both point along the same line
v1 = np.array([1, 2])
v2 = np.array([2, 4])   # v2 = 2 * v1 — redundant
# Any combination a*v1 + b*v2 = (a + 2b) * v1 — still on the line through v1
print("span{v1, v2} is just the line y = 2x")


## Linear independence and dependence

A set of vectors `{v_1, v_2, ..., v_k}` is **linearly independent** if the only way to write the zero vector as a linear combination is the trivial one:

$$
c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 + \dots + c_k \mathbf{v}_k = \mathbf{0}
\quad \Longrightarrow \quad c_1 = c_2 = \dots = c_k = 0
$$

If you can find some non-zero combination that hits zero, the set is **linearly dependent** — and at least one of the vectors is **redundant** (it can be written as a combination of the others).

The intuition: in our `v1 = [1, 2]`, `v2 = [2, 4]` example, we have `2 v1 + (-1) v2 = 0`. The coefficients are not all zero, so the set is dependent. Geometrically, `v2` is just `v1` scaled — it adds nothing new.

**The shortcut.** Stack the vectors as columns of a matrix and compute the **rank**. The rank counts how many linearly independent vectors are in the set. If `rank == k`, the set is independent. If `rank < k`, it is dependent and `k - rank` vectors are redundant.

Why this matters in ML: when two features in your dataset are linearly dependent (one is a linear combination of others), your model has no way to assign unique weights to them. `X^T X` becomes singular, ridge regression's `λI` term gets invented to fix exactly this problem, and PCA's whole job is to find a non-redundant set of directions.


In [ ]:
# Linearly independent — two non-parallel vectors in R^2
A_indep = np.column_stack([np.array([1, 0]), np.array([0, 1])])
print("Independent set, rank =", np.linalg.matrix_rank(A_indep))

# Linearly dependent — v2 is twice v1
A_dep = np.column_stack([np.array([1, 2]), np.array([2, 4])])
print("Dependent set,   rank =", np.linalg.matrix_rank(A_dep))

# Three vectors in R^2 — must be dependent (you can have at most 2 independent vectors in R^2)
A_three = np.column_stack([np.array([1, 0]), np.array([0, 1]), np.array([3, 5])])
print("Three vecs in R^2, rank =", np.linalg.matrix_rank(A_three), "<- at most 2 independent")


## Basis — the minimum spanning set

A **basis** for a vector space is a set of vectors that is

1. **Linearly independent** (no redundancy), and
2. **Spans** the whole space (you can reach every vector by combining them).

In short: a basis is the *smallest* set you need to describe the entire space.

The **standard basis** of `R^2` is `e_1 = [1, 0]`, `e_2 = [0, 1]`. Any vector `[a, b]` is `a · e_1 + b · e_2`. The standard basis of `R^n` follows the same pattern: `n` vectors, each with a single 1 and the rest zeros.

But the standard basis is just one choice. **Any** linearly independent set of `n` vectors in `R^n` is a basis. For example, `b_1 = [2, 1]` and `b_2 = [1, 3]` is also a basis for `R^2` — they are independent (not parallel) and they span the plane.

**Coordinates depend on the basis.** When you write a vector as `[3, 4]`, you implicitly mean "3 of `e_1` plus 4 of `e_2`." In a different basis, the same point has different coordinates. PCA, Fourier transforms, and wavelet transforms are all really "express the data in a smarter basis."

This is one of the most important moves in ML: keep the data, change the basis.


In [ ]:
# Express the vector [3, 4] in two different bases of R^2

x = np.array([3, 4])

# Basis 1: standard — coordinates are just (3, 4)
print("Standard basis coords of x:", x)

# Basis 2: b1 = [2, 1], b2 = [1, 3]
# Find coords c1, c2 such that c1*b1 + c2*b2 = x
# That is, solve B @ c = x, where B is the matrix whose columns are b1, b2
B = np.column_stack([np.array([2, 1]), np.array([1, 3])])
c = np.linalg.solve(B, x)
print(f"New basis coords:           {c}")
print(f"Check c1*b1 + c2*b2:        {B @ c}  (matches x)")


## Dimension

The **dimension** of a vector space is the number of vectors in any basis for it. *Any* basis — every basis of `R^2` has exactly 2 vectors, every basis of `R^3` has exactly 3, every basis of `R^n` has exactly `n`. Dimension is an intrinsic property of the space, not of a particular basis.

Some useful facts that follow:

- You **need at least `n` vectors** to span `R^n`. Fewer than `n`, and your span is a proper subspace (a line, plane, etc.).
- You **can have at most `n` independent vectors** in `R^n`. Any larger set must contain a linear dependence.
- A set of exactly `n` vectors in `R^n` is a basis if and only if it is linearly independent (equivalently: `det ≠ 0`).

In ML, "dimension" often gets used loosely:

- **Ambient dimension** — the number of features in your data. MNIST images have ambient dimension 784.
- **Intrinsic dimension** — the dimension of the (often much smaller) subspace where the data *actually* lives. MNIST images live near a much lower-dimensional manifold; PCA tries to estimate that.

The gap between ambient and intrinsic dimension is *why* dimensionality reduction works at all.


## Subspaces — vector spaces inside vector spaces

A **subspace** of `R^n` is a subset that is itself a vector space — i.e., it is closed under addition and scalar multiplication. Geometrically, a subspace of `R^n` is always one of:

- The **trivial subspace** containing just the zero vector — dimension 0.
- A **line through the origin** — dimension 1.
- A **plane through the origin** — dimension 2.
- ... and so on up to ...
- All of `R^n` itself — dimension `n`.

**Subspaces must pass through the origin.** A line that doesn't go through the origin is not a subspace, because scaling any point on it by zero gives the origin, which is then not on the line.

Two subspaces matter so much that they get the whole of notebook 5:

- The **column space** of a matrix `A` — the span of its columns. This is the set of all `b` for which `Ax = b` has a solution.
- The **null space** of a matrix `A` — the set of all `x` such that `Ax = 0`. This measures how much of `R^n` the matrix collapses to zero.

Together with two more subspaces, these form Gilbert Strang's *four fundamental subspaces*, the cleanest summary of what a matrix does.


In [ ]:
# A 2D subspace of R^3: the plane spanned by e1 and e2 (the xy-plane)
# A vector lies in this subspace iff its z-component is 0.

v_in  = np.array([3, 4, 0])   # in the xy-plane
v_out = np.array([3, 4, 1])   # not in the xy-plane

basis_xy = np.column_stack([np.array([1, 0, 0]), np.array([0, 1, 0])])

def lies_in_span(v, basis):
    # Solve basis @ c = v in the least-squares sense; if residual ~ 0, v is in the span
    c, *_ = np.linalg.lstsq(basis, v, rcond=None)
    return np.allclose(basis @ c, v), c

inside,  c_in  = lies_in_span(v_in,  basis_xy)
outside, c_out = lies_in_span(v_out, basis_xy)
print(f"v_in  = {v_in}: in xy-plane? {inside},  coords in basis = {c_in}")
print(f"v_out = {v_out}: in xy-plane? {outside}, coords in basis = {c_out}  (best approximation only)")


## Where this appears in ML

Span, independence, basis, and dimension underlie every dimensionality-reduction and feature-engineering decision you will make.

- **Feature redundancy / multicollinearity.** If two features in your dataset are linearly dependent, you have a basis problem: your design matrix has columns that don't bring new information. `X^T X` becomes singular, and you cannot solve the normal equations directly — hence ridge regression's `λI` fix.
- **PCA.** Finds a new orthonormal **basis** for your data, ordered by variance. The first `k` basis vectors span the best `k`-dimensional **subspace** approximation to your dataset.
- **Word and image embeddings.** Each embedding is a vector in some `R^d`. The embedding *space* is a vector space, and similar concepts cluster into low-dimensional subspaces inside it.
- **Latent variables / bottleneck layers.** Autoencoders compress high-dimensional input into a low-dimensional latent code — they are explicitly learning a basis for the data's intrinsic subspace.
- **Rank-based regularization.** Low-rank matrix factorization (recommender systems, LoRA in LLM fine-tuning) constrains weight matrices to have small rank, i.e., to lie in a low-dimensional subspace.
- **Underdetermined systems.** When `# features > # samples`, the normal equations have infinitely many solutions — there is a whole subspace of valid `w`'s. Regularization picks one.

Next notebook: **matrices** — once we have vectors and the vector spaces they live in, a matrix becomes a *function* between those spaces. That single perspective unlocks the rest of linear algebra.
